# 03 — Structured ensemble (D8) · axes B / C / D · **gates G7 + G12**

Third of four (spec §9, step 3). Axis B = band cutoff × {0.5, 1, 2}; axis C = leave-one-out
**by name** (all parts of a name dropped together, `no_link` parts included — independent in the
graph but sharing the name's realisation risk); axis D = β ∈ {1.5, 2.5, 4.0}. One cached CWD
set serves every member; serial with memmaps on purpose. Resumable: a member is done when its
`member.json` exists.

**Axis C is the one to read closely.** The decision after this notebook — whether to open
Phase 7 (climate-modified routing) at all — is made on axis-C stability; it does NOT block
notebook 04, which is climate-free by construction (D14).


In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/northern_connectivity/, below the repo root where config.py
# and the corridor engine modules (corridors_prep / corridor_graph / corridors_core /
# corridors_ensemble) sit. Same bootstrap pattern as analyses/y2y/.
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
import corridors_ensemble as ce
for _m in (config, cp, cg, cc, ce):
    importlib.reload(_m)

KEY = "north"


In [ ]:
# ---- Re-attach to the run created by 02_calibrate_baseline ------------------
# One run spans notebooks 02-04: 02 created it via cc.start(); here cc.load() reads ONLY the run
# dir (run_config.json + the H7 copies), then the cached CWD + graph are re-derived -- the CWD
# stage is a cache HIT (seconds), the network rebuild is minutes. cwd_cutoff_abs was written into
# run_config.json by cc.set_cutoff in notebook 02, so nothing here depends on config.py.
RUN = "v2_run001"                       # <-- the run to continue

A = cc.load(config.RESULTS_DIR / "corridors_north" / RUN)
cc.resistance(A)
cc.cost_distances(A)                    # cache HIT
cc.corridor_network(A, verbose=False)
A


## Gates — **G7** (cache reuse is sound) and **G12** (47 distinct members by config-hash)


In [ ]:
ce.gate_g7(A)
ce.gate_g12(A)


## Run every member (resumable)


In [ ]:
ce.run(A)


## Collect — attribution surface + per-axis rasters (**D15**)

Written as `ensemble_attribution.tif` (+ `attribution_<axis>.tif`), used for the robust-core
threshold (0.9) and attribution only. NEVER labelled "frequency": ~42 of 47 members are
leave-one-out, so the fraction is "share of dropped names that didn't matter", not a
near-optimal sampling frequency — that product is notebook 04's near-optimality surface.


In [ ]:
ce.collect(A)
